# ✈️ AOG 자재 수급 대시보드

아래 셀 **하나만** 실행하면 대시보드가 셀 출력 안에 바로 표시됩니다 (Colab 자동 인라인 표시, 외부 터널/IP 조회 불필요). 사용법과 테스트 시나리오는 저장소의 `README.md`를 참고하세요.

In [ ]:
# ============================================================================
# ✈️ AOG(Aircraft on Ground) 자재 수급 대시보드
# ----------------------------------------------------------------------------
# Google Colab에서 이 셀 하나만 실행하면 됩니다.
# UI 프레임워크: Gradio (Colab이 자동으로 인식해 셀 출력 안에 인라인으로 띄워줍니다.
#                제3자 터널(localtunnel 등)이나 외부 IP 조회가 전혀 필요 없습니다.)
# ============================================================================

!pip install -q gradio

import gradio as gr

# ----------------------------------------------------------------------------
# 1. 더미 데이터 (Dummy Data)
#    실제 운영 시스템에서는 이 부분이 사내 재고관리시스템(IMS) API 조회로 대체됩니다.
# ----------------------------------------------------------------------------

AIRCRAFT_TYPES = ["A330-300", "B777-300ER", "B737-800", "A321neo"]
AIRPORTS = ["ICN", "GMP", "NRT", "CDG", "JFK", "SIN"]

# 자재 넘버 드롭다운에 노출할 전체 카탈로그 (README 테스트 시나리오와 1:1로 대응)
PART_NUMBER_CATALOG = [
    "OXY-GEN-A330-15",       # 시나리오 1: FAK에서 해결
    "BRK-B777-CARBON-01",    # 시나리오 2: Allocation에서 해결
    "HYD-PUMP-737-11",       # 시나리오 3: Pooling 파트너사에서 해결
    "FMS-A321-NEO-09",       # 시나리오 4: Hand-carry까지 가는 케이스
    "WHL-NLG-737-22",        # 참고: 동일기종 타항공사(5단계)에서 해결되는 케이스
    "CAB-DOOR-ACT-321-03",   # 참고: Main Station 타항공사(4단계)에서 해결되는 케이스
]

# 1단계: 당사 FAK(Fly Away Kit) 재고
FAK_STOCK = [
    {"aircraft_type": "A330-300", "part_number": "OXY-GEN-A330-15", "qty": 2, "location": "ICN FAK 창고 A-12"},
    {"aircraft_type": "B737-800", "part_number": "SMK-DET-737-04", "qty": 3, "location": "ICN FAK 창고 B-05"},
]

# 2단계: 당사 Allocation 재고 (본사 배정 재고)
ALLOCATION_STOCK = [
    {"aircraft_type": "B777-300ER", "part_number": "BRK-B777-CARBON-01", "qty": 1, "location": "ICN 본사 Allocation 창고"},
    {"aircraft_type": "A330-300", "part_number": "FUEL-NOZ-A330-07", "qty": 4, "location": "ICN 본사 Allocation 창고"},
]

# 3단계: Pooling 계약 파트너사 및 보유 재고
POOLING_PARTNERS = {
    "SIA Engineering (싱가포르)": {
        "contact": "+65-6541-2000 / poolingdesk@siae.com.sg",
        "stock": [
            {"aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "qty": 1, "location": "SIN 창고"},
        ],
    },
    "Lufthansa Technik (독일)": {
        "contact": "+49-40-5070-5553 / pooling@lht.dlh.de",
        "stock": [
            {"aircraft_type": "A330-300", "part_number": "IDG-A330-001", "qty": 1, "location": "FRA 창고"},
        ],
    },
    "ANA Base Maintenance (일본)": {
        "contact": "+81-3-6735-1111 / pooling@ana.co.jp",
        "stock": [],
    },
}

# 4단계: 공항별 Main Station 운영 항공사 (해당 공항을 허브/메인기지로 쓰는 항공사)
MAIN_STATION_AIRLINES = {
    "ICN": [{"airline": "아시아나항공", "contact": "02-2669-8000"}],
    "GMP": [{"airline": "제주항공", "contact": "02-2015-1000"}],
    "NRT": [{"airline": "ANA", "contact": "+81-3-6735-1000"}],
    "CDG": [{"airline": "Air France", "contact": "+33-1-4356-7890"}],
    "JFK": [{"airline": "Delta Air Lines", "contact": "+1-800-221-1212"}],
    "SIN": [{"airline": "Singapore Airlines", "contact": "+65-6223-8888"}],
}
# (공항, 자재넘버) 조합으로 실제 대여 가능한 재고가 있는 경우만 기재 (없으면 "대여 불가"로 처리)
OTHER_AIRLINE_STATION_STOCK = {
    ("GMP", "CAB-DOOR-ACT-321-03"): {"airline": "제주항공", "qty": 1, "location": "GMP", "contact": "02-2015-1000"},
}

# 5단계: 동일 기종을 운영 중인 타 항공사
FLEET_OPERATORS = {
    "A330-300": ["아시아나항공", "Cathay Pacific", "China Eastern"],
    "B777-300ER": ["아시아나항공", "Emirates", "ANA"],
    "B737-800": ["제주항공", "티웨이항공", "Ryanair"],
    "A321neo": ["에어부산", "IndiGo", "Wizz Air"],
}
# (기종, 자재넘버) 조합으로 실제 대여 가능한 재고가 있는 경우만 기재
OTHER_AIRLINE_FLEET_STOCK = {
    ("B737-800", "WHL-NLG-737-22"): {"airline": "티웨이항공", "qty": 1, "location": "ICN", "contact": "1688-8686"},
}

STEP_NAMES = {
    1: "1단계 · FAK(Fly Away Kit) 자재 재고 확인",
    2: "2단계 · Allocation 자재 재고 확인",
    3: "3단계 · Pooling 파트너사 재고 확인",
    4: "4단계 · 해당 공항 Main Station 운영 항공사 대여 요청",
    5: "5단계 · 동일 기종 운영 타 항공사 대여 요청",
    6: "6단계 · 본사 직원 Hand-carry 이동",
}
TOTAL_STEPS = 6


# ----------------------------------------------------------------------------
# 2. 단계별 재고/대여 조회 로직
#    evaluate_step()은 "현재 단계에서 자재를 확보했는가?"만 판단하고,
#    실제 다음 단계로 넘어갈지 여부는 UI의 버튼 클릭(사용자 확인)에 맡긴다.
# ----------------------------------------------------------------------------

def evaluate_step(step: int, aircraft_type: str, part_number: str, airport: str):
    """주어진 단계(step)에서 자재 확보 여부와 사용자에게 보여줄 메시지를 반환한다."""

    if step == 1:
        hit = next((r for r in FAK_STOCK
                    if r["aircraft_type"] == aircraft_type and r["part_number"] == part_number), None)
        if hit:
            return True, (f"✅ **FAK 재고 확인됨**\n"
                           f"- 위치: {hit['location']}\n- 가용 수량: {hit['qty']}개\n\n"
                           f"당사 FAK 재고로 즉시 조치 가능합니다.")
        return False, "❌ FAK(Fly Away Kit)에는 해당 자재 재고가 없습니다."

    if step == 2:
        hit = next((r for r in ALLOCATION_STOCK
                    if r["aircraft_type"] == aircraft_type and r["part_number"] == part_number), None)
        if hit:
            return True, (f"✅ **Allocation 재고 확인됨**\n"
                           f"- 위치: {hit['location']}\n- 가용 수량: {hit['qty']}개\n\n"
                           f"당사 Allocation 재고로 조치 가능합니다.")
        return False, "❌ 당사 Allocation 재고에도 해당 자재가 없습니다."

    if step == 3:
        for partner, info in POOLING_PARTNERS.items():
            hit = next((r for r in info["stock"]
                        if r["aircraft_type"] == aircraft_type and r["part_number"] == part_number), None)
            if hit:
                return True, (f"✅ **Pooling 파트너사 재고 확인됨 — {partner}**\n"
                               f"- 위치: {hit['location']}\n- 가용 수량: {hit['qty']}개\n"
                               f"- 연락처: {info['contact']}\n\n"
                               f"Pooling 계약에 따라 대여 요청이 가능합니다.")
        checked = ", ".join(POOLING_PARTNERS.keys())
        return False, f"❌ 조회한 Pooling 파트너사({checked}) 중 해당 자재 재고를 보유한 곳이 없습니다."

    if step == 4:
        hit = OTHER_AIRLINE_STATION_STOCK.get((airport, part_number))
        operators = MAIN_STATION_AIRLINES.get(airport, [])
        op_names = ", ".join(o["airline"] for o in operators) if operators else "등록된 항공사 없음"
        if hit:
            return True, (f"✅ **Main Station 타 항공사 재고 확인됨 — {hit['airline']}**\n"
                           f"- 위치: {hit['location']}\n- 가용 수량: {hit['qty']}개\n"
                           f"- 연락처: {hit['contact']}\n\n"
                           f"{airport}을 Main Station으로 운영 중인 항공사로부터 대여 가능합니다.")
        return False, (f"❌ {airport}을 Main Station으로 운영하는 항공사({op_names})에 문의했으나 "
                        f"재고를 보유한 곳이 없습니다.")

    if step == 5:
        hit = OTHER_AIRLINE_FLEET_STOCK.get((aircraft_type, part_number))
        operators = FLEET_OPERATORS.get(aircraft_type, [])
        op_names = ", ".join(operators) if operators else "등록된 항공사 없음"
        if hit:
            return True, (f"✅ **동일 기종 운영 타 항공사 재고 확인됨 — {hit['airline']}**\n"
                           f"- 위치: {hit['location']}\n- 가용 수량: {hit['qty']}개\n"
                           f"- 연락처: {hit['contact']}\n\n"
                           f"{aircraft_type} 기종을 운영 중인 타 항공사로부터 대여 가능합니다.")
        return False, (f"❌ {aircraft_type} 운영 항공사({op_names})에 문의했으나 재고를 보유한 곳이 없습니다.")

    if step == 6:
        # Hand-carry는 최후의 수단으로, 본사 재고 확보를 전제로 항상 조치 가능하다고 가정한다.
        return True, ("🧳 **본사 직원 Hand-carry 이동 확정**\n"
                       "- 본사 자재관리팀이 자재를 확보해 가장 빠른 항공편 승무원/출장자 편으로 발송합니다.\n"
                       "- 담당 부서: 자재관리팀 (02-XXXX-XXXX)\n\n"
                       "모든 사전 단계에서 재고를 찾지 못해 최종 수단으로 조치합니다.")

    raise ValueError(f"알 수 없는 단계: {step}")


# ----------------------------------------------------------------------------
# 3. 화면 렌더링 헬퍼
#    상태(state)를 받아 Gradio 컴포넌트 업데이트 값들을 일관되게 만들어 낸다.
# ----------------------------------------------------------------------------

def render(state):
    step = state["step"]
    found, _ = state["last_result"]

    step_label = f"### 진행 단계 {step}/{TOTAL_STEPS} — {STEP_NAMES[step]}"

    status_icon = "🟢 자재 확보" if found else "🟡 자재 미확보"
    result_text = f"**{status_icon}**\n\n{state['last_message']}"

    log_text = "\n\n---\n\n".join(state["log"])

    is_last_step = step >= TOTAL_STEPS
    proceed_visible = (not state["finished"]) and (not is_last_step)
    resolve_visible = not state["finished"]
    resolve_label = ("✅ Hand-carry 조치 확정 (프로세스 종료)" if is_last_step
                      else "✅ 현재 단계에서 조치 (프로세스 종료)")

    return (
        state,
        gr.update(value=step_label, visible=True),
        gr.update(value=result_text, visible=True),
        gr.update(visible=proceed_visible),
        gr.update(value=resolve_label, visible=resolve_visible),
        gr.update(value=log_text, visible=True),
    )


def _append_log(state, header, message):
    state["log"].append(f"**[{header}]**\n\n{message}")


# ----------------------------------------------------------------------------
# 4. 이벤트 핸들러 (버튼 클릭 시 실행되는 함수들)
# ----------------------------------------------------------------------------

def start_case(aircraft_type, part_number, airport):
    if not aircraft_type or not part_number or not airport:
        empty_state = {"step": 0, "log": [], "finished": True, "last_result": (False, ""), "last_message": ""}
        return (
            empty_state,
            gr.update(value="⚠️ 항공기 기종 / 자재 넘버 / 공항을 모두 선택한 뒤 시작해주세요.", visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
        )

    state = {
        "aircraft_type": aircraft_type,
        "part_number": part_number,
        "airport": airport,
        "step": 1,
        "log": [],
        "finished": False,
        "last_result": (False, ""),
        "last_message": "",
    }
    state["log"].append(
        f"### 🚨 AOG 케이스 시작\n- 기종: **{aircraft_type}**\n- 자재 넘버: **{part_number}**\n- 발생 공항: **{airport}**"
    )

    found, message = evaluate_step(1, aircraft_type, part_number, airport)
    state["last_result"] = (found, message)
    state["last_message"] = message
    _append_log(state, STEP_NAMES[1], message)

    return render(state)


def proceed(state):
    if state.get("finished") or state["step"] >= TOTAL_STEPS:
        return render(state)

    next_step = state["step"] + 1
    found, message = evaluate_step(next_step, state["aircraft_type"], state["part_number"], state["airport"])
    state["step"] = next_step
    state["last_result"] = (found, message)
    state["last_message"] = message
    _append_log(state, STEP_NAMES[next_step], message)

    return render(state)


def resolve_here(state):
    if state.get("finished"):
        return render(state)

    state["finished"] = True
    step = state["step"]
    _append_log(
        state,
        "프로세스 종료",
        f"담당자가 **{STEP_NAMES[step]}** 단계에서 조치를 완료하여 AOG 프로세스를 종료합니다. 🏁",
    )
    return render(state)


# ----------------------------------------------------------------------------
# 5. Gradio 대시보드 UI 구성
# ----------------------------------------------------------------------------

with gr.Blocks(title="AOG 자재 수급 대시보드") as demo:
    gr.Markdown(
        "# ✈️ AOG(Aircraft on Ground) 자재 수급 대시보드\n"
        "AOG 상황 발생 시 정해진 절차(FAK → Allocation → Pooling → Main Station 타사 → 동일기종 타사 → Hand-carry)에 따라 "
        "자재 확보 여부를 단계별로 확인합니다. 각 단계 결과를 확인한 뒤 **다음 단계 진행** 또는 "
        "**현재 단계에서 조치(종료)** 를 직접 선택해야 다음으로 넘어갑니다."
    )

    with gr.Row():
        aircraft_dd = gr.Dropdown(choices=AIRCRAFT_TYPES, label="항공기 기종 (Aircraft Type)")
        part_dd = gr.Dropdown(choices=PART_NUMBER_CATALOG, label="자재 넘버 (Part Number)", allow_custom_value=True)
        airport_dd = gr.Dropdown(choices=AIRPORTS, label="AOG 발생 공항 (Station)")

    start_btn = gr.Button("🚨 AOG 프로세스 시작", variant="primary")

    step_md = gr.Markdown(visible=False)
    result_md = gr.Markdown(visible=False)

    with gr.Row():
        proceed_btn = gr.Button("▶ 다음 단계 진행", visible=False)
        resolve_btn = gr.Button("✅ 현재 단계에서 조치 (프로세스 종료)", visible=False)

    gr.Markdown("### 📋 처리 이력 (Audit Log)")
    log_md = gr.Markdown(visible=False)

    state = gr.State({})

    outputs = [state, step_md, result_md, proceed_btn, resolve_btn, log_md]

    start_btn.click(start_case, inputs=[aircraft_dd, part_dd, airport_dd], outputs=outputs)
    proceed_btn.click(proceed, inputs=[state], outputs=outputs)
    resolve_btn.click(resolve_here, inputs=[state], outputs=outputs)

# share=True를 쓰지 않으므로 gradio.live 같은 제3자 터널이 필요 없다.
# Colab에서 실행하면 자동으로 셀 출력 안에 인라인으로 대시보드가 표시된다.
demo.launch(debug=False)
